# Date, Time & Serialization (CSV & JSON) (5+ Years Interview Guide)
Exhaustive revision guide to datetime parsing (strptime), formatting (strftime), timedelta arithmetic, JSON serialization (dumps/loads), and CSV streaming on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Temporal Parsing & Formatting**: Dedicated cell for `datetime.strptime()` and `.strftime()`.
- **Date Arithmetic**: Dedicated cell for `timedelta(days=..., hours=...)`.
- **JSON Serialization**: Dedicated cell for `json.dumps()`, `json.loads()`, `json.dump()`, and `json.load()`.
- **CSV Handling**: Dedicated cell for `csv.DictReader` and `csv.DictWriter`.

This interactive revision guide uses `data/raw_transactions.csv` with individual dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import json
import re
import collections
from datetime import datetime, timedelta

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from data/raw_transactions.csv


### Datetime Parsing & Formatting: `strptime` and `strftime`
**Explanation**: `strptime(date_string, format)` parses strings into `datetime` objects. `strftime(format)` formats `datetime` objects into custom string timestamps.

**Syntax**: `datetime.strptime('2025-01-01', '%Y-%m-%d')` / `dt.strftime('%B %d, %Y')`

In [2]:
sample_date_str = transactions[0]['transaction_date']
try:
    dt_obj = datetime.strptime(sample_date_str, '%Y-%m-%d %H:%M:%S')
except ValueError:
    dt_obj = datetime.strptime(sample_date_str, '%m-%d-%Y')
print('Parsed Datetime Object:', dt_obj)
print('Formatted String (strftime):', dt_obj.strftime('%A, %d %B %Y at %I:%M %p'))

Parsed Datetime Object: 2025-07-06 00:00:00
Formatted String (strftime): Sunday, 06 July 2025 at 12:00 AM


### Date Arithmetic with `timedelta`
**Explanation**: Performs temporal additions and subtractions across days, hours, and seconds.

**Syntax**: `dt + timedelta(days=30)`

In [3]:
settlement_date = dt_obj + timedelta(days=3)
chargeback_deadline = dt_obj + timedelta(days=90)
print('Transaction Date:', dt_obj.date())
print('Settlement Date (T+3):', settlement_date.date())
print('Chargeback Deadline (T+90):', chargeback_deadline.date())

Transaction Date: 2025-07-06
Settlement Date (T+3): 2025-07-09
Chargeback Deadline (T+90): 2025-10-04


### JSON Serialization: `dumps` and `loads`
**Explanation**: `json.dumps()` serializes Python dicts/lists to JSON strings. `json.loads()` deserializes JSON strings back to Python data structures.

**Syntax**: `json.dumps(obj, indent=2)` / `json.loads(json_str)`

In [4]:
sample_payload = transactions[0]
json_str = json.dumps(sample_payload, indent=2)
print('Serialized JSON String (sample):\n', json_str[:150], '...')
restored_dict = json.loads(json_str)
print('Restored Transaction ID:', restored_dict['transaction_id'])

Serialized JSON String (sample):
 {
  "transaction_id": "TX110686",
  "customer_id": "C82845",
  "merchant_id": "M2697",
  "transaction_amount": "1216.33",
  "card_type": "Visa",
  "tr ...
Restored Transaction ID: TX110686


### CSV Serialization: `csv.DictReader` and `csv.DictWriter`
**Explanation**: Streams tabular transaction records directly to and from CSV files with header management.

**Syntax**: `csv.DictWriter(f, fieldnames=...)`

In [5]:
os.makedirs('scratch', exist_ok=True)
out_csv_path = 'scratch/exported_tx.csv'
with open(out_csv_path, mode='w', newline='', encoding='utf-8') as f:
    fieldnames = ['transaction_id', 'transaction_amount', 'card_type']
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(transactions[:3])

with open(out_csv_path, mode='r', encoding='utf-8') as f:
    print('Exported CSV Contents:\n', f.read().strip())

Exported CSV Contents:
 transaction_id,transaction_amount,card_type
TX110686,1216.33,Visa
TX107170,324.99,MasterCard
TX108328,136.66,Discover


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Custom JSON Encoding for Datetime & Decimal Objects
**Explanation**: Implement a custom `json.JSONEncoder` subclass to handle non-serializable objects (`datetime`, `Decimal`) without raising TypeErrors.

**Syntax**: `class CustomEncoder(json.JSONEncoder): def default(self, obj): ...`

In [6]:
class CustomFintechEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()
        return super().default(obj)

payload = {'tx_id': 'TX_100', 'timestamp': datetime.now()}
print('Custom JSON Encoded:', json.dumps(payload, cls=CustomFintechEncoder))

Custom JSON Encoded: {"tx_id": "TX_100", "timestamp": "2026-08-30T14:55:38.253172"}
